# Spotify Music Explorer

This notebook explores the Spotify Tracks dataset and indexes the cleaned data into Elasticsearch for search, filtering, and aggregation experiments.


In [1]:
# Import the libraries used for data exploration and Elasticsearch access.
import pandas as pd
from elasticsearch import Elasticsearch


# Load the Spotify Tracks dataset from the local raw data folder.
spotify_data = "../data/raw/dataset.csv"


df = pd.read_csv(spotify_data)


## 1. Load the Dataset

Read the Kaggle Spotify Tracks CSV into a pandas dataframe so the data can be inspected before indexing.


In [2]:
df.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## 2. Inspect the Raw Data

Check the dataframe structure, available columns, and missing values before deciding what to index.


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  str    
 2   artists           113999 non-null  str    
 3   album_name        113999 non-null  str    
 4   track_name        113999 non-null  str    
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          114000 non-null  bool   
 8   danceability      114000 non-null  float64
 9   energy            114000 non-null  float64
 10  key               114000 non-null  int64  
 11  loudness          114000 non-null  float64
 12  mode              114000 non-null  int64  
 13  speechiness       114000 non-null  float64
 14  acousticness      114000 non-null  float64
 15  instrumentalness  114000 non-null  float64
 16  liveness          114000 non-nu

In [4]:
df.columns

Index(['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name',
       'popularity', 'duration_ms', 'explicit', 'danceability', 'energy',
       'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature',
       'track_genre'],
      dtype='str')

In [5]:
df.isna().sum().sort_values(ascending=False)

artists             1
track_name          1
album_name          1
Unnamed: 0          0
track_id            0
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64

## 3. Clean the Dataset

Remove unused columns and rows with missing key text fields so Elasticsearch receives consistent documents.


In [6]:
# Remove columns that are not needed for search/indexing in this project.
df = df.drop(columns=["Unnamed: 0", "time_signature"])

df.columns


Index(['track_id', 'artists', 'album_name', 'track_name', 'popularity',
       'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness',
       'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'track_genre'],
      dtype='str')

## 4. Quick Dataset Exploration

Run a few simple dataframe checks to understand genres, artists, and example records before building search queries.


In [7]:
df[df["track_genre"] == "k-pop"]


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,track_genre
65000,3hkC9EHFZNQPXrtl8WPHnX,Alka Yagnik;Arijit Singh,Tamasha,Agar Tum Saath Ho,73,341054,False,0.562,0.519,3,-8.744,1,0.0326,0.5570,0.000268,0.1720,0.415,122.925,k-pop
65001,45OX2jjEw1l7lOFJfDP9fv,LISA,LALISA,MONEY,78,168227,False,0.826,0.553,1,-10.121,0,0.2340,0.1630,0.000041,0.1350,0.400,140.037,k-pop
65002,5aucVLKiumD89mxVCB4zvS,Crush;j-hope,Rush Hour,Rush Hour (Feat. j-hope of BTS),83,177302,False,0.738,0.714,0,-5.235,1,0.2490,0.1570,0.000000,0.3100,0.740,95.035,k-pop
65003,1R0hxCA5R7z5TiaXBZR7Mf,JENNIE,SOLO,SOLO,73,169566,False,0.752,0.642,3,-5.165,0,0.0869,0.0861,0.000000,0.0953,0.388,95.043,k-pop
65004,4a9tbd947vo9K8Vti9JwcI,BTS;Halsey,MAP OF THE SOUL : PERSONA,Boy With Luv (feat. Halsey),80,229773,False,0.645,0.862,11,-4.761,0,0.0845,0.0933,0.000000,0.1930,0.803,119.947,k-pop
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65995,6yN6AP2qkoGIjhF3VzUOkO,Alka Yagnik;Kumar Sanu,Kahin Pyaar Na Ho Jaaye (Original Motion Pictu...,Kahin Pyaar Na Ho Jaaye,44,318546,False,0.622,0.748,1,-7.730,1,0.0437,0.1610,0.000000,0.1000,0.725,128.844,k-pop
65996,5kkgQsFJY5MVXChIfY0bl9,Yuvan Shankar Raja;KK;Sadhana Sargam,Dass,Sakka Podu,42,257743,False,0.806,0.633,0,-4.674,0,0.0487,0.2890,0.000000,0.1690,0.740,103.060,k-pop
65997,4jUEHIrc443f743JbyLN0y,BLACKPINK,BLACKPINK IN YOUR AREA (Japanese Version),DDU-DU DDU-DU - Japanese Version,61,209493,False,0.703,0.845,11,-3.479,0,0.0682,0.0296,0.000000,0.2130,0.428,139.936,k-pop
65998,19MRAxznRvejs0DNu8MhoC,Yuvan Shankar Raja;Priya Hemesh,Kazhugoo (Original Motion Picture Soundtrack),Aathadi Manasudhan,43,309666,False,0.639,0.476,3,-7.675,0,0.0430,0.1380,0.052300,0.0865,0.353,160.093,k-pop


In [8]:
df[df["artists"] == "BTS"]

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,track_genre
65008,5FVbvttjEvQ8r2BgUcJgNg,BTS,BE,Life Goes On,79,207481,False,0.566,0.716,1,-5.733,1,0.0424,0.00691,0.0,0.3700,0.450,81.068,k-pop
65011,5QDLhrAOJJdNAmCTJ8xMyW,BTS,BE,Dynamite,85,199053,False,0.746,0.765,6,-4.410,0,0.0993,0.01120,0.0,0.0936,0.737,114.044,k-pop
65018,6m1TWFMeon7ai9XLOzdbiR,BTS,Love Yourself 轉 'Tear',FAKE LOVE,76,242333,False,0.557,0.719,2,-4.515,0,0.0371,0.00267,0.0,0.3060,0.345,77.502,k-pop
65024,5YMXGBD6vcYP7IolemyLtK,BTS,Love Yourself 結 'Answer',Euphoria,78,228615,False,0.637,0.799,2,-4.518,1,0.0338,0.39400,0.0,0.0921,0.570,104.996,k-pop
65030,69xohKu8C1fsflYAiSNbwM,BTS,Proof,Run BTS,84,204939,False,0.724,0.818,8,-3.747,1,0.1680,0.02010,0.0,0.0358,0.696,77.004,k-pop
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81562,5FVbvttjEvQ8r2BgUcJgNg,BTS,BE,Life Goes On,79,207481,False,0.566,0.716,1,-5.733,1,0.0424,0.00691,0.0,0.3700,0.450,81.068,pop
81589,6jjYDGxVJsWS0a5wlVF5vS,BTS,Proof,Butter,83,164952,False,0.759,0.459,8,-5.187,1,0.0948,0.00323,0.0,0.0788,0.695,109.997,pop
81783,5YMXGBD6vcYP7IolemyLtK,BTS,Love Yourself 結 'Answer',Euphoria,78,228615,False,0.637,0.799,2,-4.518,1,0.0338,0.39400,0.0,0.0921,0.570,104.996,pop
81831,3XYRV7ZSHqIRDG87DKTtry,BTS,Permission to Dance,Permission to Dance,80,187585,False,0.702,0.741,9,-5.330,1,0.0427,0.00544,0.0,0.3370,0.646,124.925,pop


In [9]:
# Drop the few rows that contain missing artist, album, or track values.
df = df.dropna()


In [10]:
df.isna().sum().sort_values(ascending=False)


track_id            0
artists             0
album_name          0
track_name          0
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
track_genre         0
dtype: int64

In [11]:
df.iloc[0]

track_id            5SuOikwiRyPMVoIQDJUgSV
artists                        Gen Hoshino
album_name                          Comedy
track_name                          Comedy
popularity                              73
duration_ms                         230666
explicit                             False
danceability                         0.676
energy                               0.461
key                                      1
loudness                            -6.746
mode                                     0
speechiness                          0.143
acousticness                        0.0322
instrumentalness                  0.000001
liveness                             0.358
valence                              0.715
tempo                               87.917
track_genre                       acoustic
Name: 0, dtype: object

In [12]:
df.head(1).T

,0
track_id,5SuOikwiRyPMVoIQDJUgSV
artists,Gen Hoshino
album_name,Comedy
track_name,Comedy
popularity,73
duration_ms,230666
explicit,False
danceability,0.676
energy,0.461
key,1


## 5. Define the Elasticsearch Mapping

Create explicit field types for text search, exact filters, numeric ranges, and aggregations. `album_name.keyword` is included so album names can be grouped in aggregations.


In [13]:
# Define the Elasticsearch mapping used by the spotify_tracks index.
mapping = {
    "mappings": {
        "properties": {
            "track_name": {"type": "text"},
            "artists": {"type": "text"},
            "album_name": {
                "type": "text",
                "fields": {
                    "keyword": {"type": "keyword"}
                }
            },
            "track_genre": {"type": "keyword"},

            "popularity": {"type": "integer"},
            "duration_ms": {"type": "integer"},
            "explicit": {"type": "boolean"},

            "danceability": {"type": "float"},
            "energy": {"type": "float"},
            "loudness": {"type": "float"},
            "speechiness": {"type": "float"},
            "acousticness": {"type": "float"},
            "instrumentalness": {"type": "float"},
            "liveness": {"type": "float"},
            "valence": {"type": "float"},
            "tempo": {"type": "float"},

            "key": {"type": "integer"},
            "mode": {"type": "integer"},
        }
    }
}


## 6. Connect to Elasticsearch

Connect to the local Elasticsearch server and keep the index name centralized in `INDEX_NAME`.


In [14]:
# Keep the index name in one variable to avoid spotify_tracks / spotify-tracks mixups.
INDEX_NAME = "spotify_tracks"

client = Elasticsearch("http://localhost:9200")

client.info()


ObjectApiResponse({'name': 'fd2987be1ca5', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'wnPO1uORQHqAfIbZYudLuA', 'version': {'number': '8.5.0', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'c94b4700cda13820dad5aa74fae6db185ca5c304', 'build_date': '2022-10-24T16:54:16.433628434Z', 'build_snapshot': False, 'lucene_version': '9.4.1', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

In [15]:
dict(client.info())

{'name': 'fd2987be1ca5',
 'cluster_name': 'docker-cluster',
 'cluster_uuid': 'wnPO1uORQHqAfIbZYudLuA',
 'version': {'number': '8.5.0',
  'build_flavor': 'default',
  'build_type': 'docker',
  'build_hash': 'c94b4700cda13820dad5aa74fae6db185ca5c304',
  'build_date': '2022-10-24T16:54:16.433628434Z',
  'build_snapshot': False,
  'lucene_version': '9.4.1',
  'minimum_wire_compatibility_version': '7.17.0',
  'minimum_index_compatibility_version': '7.0.0'},
 'tagline': 'You Know, for Search'}

## 7. Create the Index

Create `spotify_tracks` with the mapping above. This setup cell should be run before indexing because recreating the index removes existing documents.


In [16]:
# Run this setup cell before indexing; running it later deletes indexed docs.
if client.indices.exists(index=INDEX_NAME):
    client.indices.delete(index=INDEX_NAME)

client.indices.create(index=INDEX_NAME, body=mapping)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'spotify_tracks'})

## 8. Check the Index and Try a Basic Search

Confirm the index exists, then use `match_all` to verify that searchable documents are returned.


In [17]:
# Non-destructive sanity check: the index should already exist from the setup cell above.
client.indices.exists(index=INDEX_NAME)

HeadApiResponse(True)

In [21]:
result = client.search(
    index=INDEX_NAME,
    query={"match_all": {}},
    size=20
)

for hit in result["hits"]["hits"]:
    source = hit["_source"]
    print(source["track_name"])

result["hits"]["hits"][1]["_source"]

IndexError: list index out of range

## 9. Bulk Index the Tracks

Convert the cleaned dataframe to bulk actions, use `track_id` as the document id, and capture bulk results so indexing errors are visible.


In [23]:
# Convert the cleaned dataframe into dictionaries for bulk indexing.
sample_docs = df.to_dict(orient="records")


In [24]:
# Build bulk actions and index documents efficiently with the Elasticsearch helper.

from elasticsearch.helpers import bulk

actions = []

for doc in sample_docs:
    actions.append({
        "_op_type": "index",
        "_index": INDEX_NAME,
        "_id": doc["track_id"],
        "_source": doc
    })

success_count, errors = bulk(
    client,
    actions,
    refresh="wait_for",
    raise_on_error=False,
)

print(f"Indexed {success_count} docs")
errors[:3]


Indexed 113999 docs


[]

## 10. Count Indexed Documents

Count the documents in Elasticsearch. This can be lower than the dataframe row count because duplicate `track_id` values overwrite existing documents.


In [25]:
client.count(index=INDEX_NAME)

ObjectApiResponse({'count': 89740, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}})

## 11. Search Examples

Try simple search queries by track title, artist, and genre.


In [26]:
query = {
    "match": {
        "track_name": "Comedy"
    }
}

client.search(index=INDEX_NAME, query=query)

ObjectApiResponse({'took': 38, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 2, 'relation': 'eq'}, 'max_score': 13.953207, 'hits': [{'_index': 'spotify_tracks', '_id': '5SuOikwiRyPMVoIQDJUgSV', '_score': 13.953207, '_source': {'track_id': '5SuOikwiRyPMVoIQDJUgSV', 'artists': 'Gen Hoshino', 'album_name': 'Comedy', 'track_name': 'Comedy', 'popularity': 73, 'duration_ms': 230666, 'explicit': False, 'danceability': 0.676, 'energy': 0.461, 'key': 1, 'loudness': -6.746, 'mode': 0, 'speechiness': 0.143, 'acousticness': 0.0322, 'instrumentalness': 1.01e-06, 'liveness': 0.358, 'valence': 0.715, 'tempo': 87.917, 'track_genre': 'songwriter'}}, {'_index': 'spotify_tracks', '_id': '0oISYk7B6GJK0euROwbfKn', '_score': 7.533188, '_source': {'track_id': '0oISYk7B6GJK0euROwbfKn', 'artists': 'Patton Oswalt', 'album_name': 'Finest Hour', 'track_name': "The Best Comedy I've Ever Seen", 'popularity': 23, 'duration_ms': 222533, 'explicit'

In [27]:
query = {
    "match": {
        "artists": "BTS"
    }
}

client.search(index=INDEX_NAME, query=query)

ObjectApiResponse({'took': 16, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 165, 'relation': 'eq'}, 'max_score': 8.624872, 'hits': [{'_index': 'spotify_tracks', '_id': '1MX0g22bQkr9HDVe37fLnN', '_score': 8.624872, '_source': {'track_id': '1MX0g22bQkr9HDVe37fLnN', 'artists': 'BTS', 'album_name': "Love Yourself 轉 'Tear'", 'track_name': '134340', 'popularity': 70, 'duration_ms': 230063, 'explicit': False, 'danceability': 0.665, 'energy': 0.687, 'key': 11, 'loudness': -6.466, 'mode': 0, 'speechiness': 0.0547, 'acousticness': 0.155, 'instrumentalness': 9.28e-05, 'liveness': 0.173, 'valence': 0.651, 'tempo': 106.023, 'track_genre': 'k-pop'}}, {'_index': 'spotify_tracks', '_id': '1rLkzFZdokhx6Wcs80uvnw', '_score': 8.624872, '_source': {'track_id': '1rLkzFZdokhx6Wcs80uvnw', 'artists': 'BTS', 'album_name': "Love Yourself 結 'Answer'", 'track_name': 'Dimple', 'popularity': 61, 'duration_ms': 196776, 'explicit': False, 'dancea

In [28]:
query = {
    "match": {
        "track_genre": "k-pop"
    }
}

client.search(index=INDEX_NAME, query=query)

ObjectApiResponse({'took': 6, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 867, 'relation': 'eq'}, 'max_score': 4.7366986, 'hits': [{'_index': 'spotify_tracks', '_id': '3r44Otr6SVUja3SYsCuhVY', '_score': 4.7366986, '_source': {'track_id': '3r44Otr6SVUja3SYsCuhVY', 'artists': 'Izzamuzzic;Julien Marchal', 'album_name': 'Shootout (Sped Up)', 'track_name': 'Shootout (Sped Up)', 'popularity': 76, 'duration_ms': 242142, 'explicit': False, 'danceability': 0.684, 'energy': 0.605, 'key': 10, 'loudness': -8.678, 'mode': 0, 'speechiness': 0.0634, 'acousticness': 0.216, 'instrumentalness': 0.941, 'liveness': 0.103, 'valence': 0.768, 'tempo': 104.647, 'track_genre': 'k-pop'}}, {'_index': 'spotify_tracks', '_id': '0Bph99neebTJZ6rsIhd0sN', '_score': 4.7366986, '_source': {'track_id': '0Bph99neebTJZ6rsIhd0sN', 'artists': 'Jubin Nautiyal;Palak Muchhal', 'album_name': 'Kaabil', 'track_name': 'Kaabil Hoon', 'popularity': 50, 'duratio

## 12. Filtering Examples

Combine text search with filters, such as genre filters and popularity ranges.


In [29]:
# Combine full-text matching with exact filters and numeric ranges.
query = {
    "bool": {
        "must": [
            {"match": {"artists": "BTS"}}
        ],
        "filter": [
            {"term": {"track_genre": "k-pop"}},
            {"range": {"popularity": {"gte": 75}}}
        ]
    }
}

results = client.search(index=INDEX_NAME, query=query)

for hit in results["hits"]["hits"]:
    song = hit["_source"]

    print(f"Track: {song['track_name']}")
    print(f"Artist: {song['artists']}")
    print(f"Album: {song['album_name']}")
    print(f"Genre: {song['track_genre']}")
    print(f"Popularity: {song['popularity']}")
    print("-" * 40)




Track: Butter
Artist: BTS
Album: Butter (Hotter, Sweeter, Cooler)
Genre: k-pop
Popularity: 78
----------------------------------------
Track: Blood Sweat & Tears
Artist: BTS
Album: Wings
Genre: k-pop
Popularity: 75
----------------------------------------
Track: For Youth
Artist: BTS
Album: Proof
Genre: k-pop
Popularity: 78
----------------------------------------
Track: Spring Day
Artist: BTS
Album: You Never Walk Alone
Genre: k-pop
Popularity: 76
----------------------------------------
Track: Filter
Artist: BTS
Album: MAP OF THE SOUL : 7
Genre: k-pop
Popularity: 76
----------------------------------------
Track: IDOL
Artist: BTS
Album: Love Yourself 結 'Answer'
Genre: k-pop
Popularity: 75
----------------------------------------
Track: Pied Piper
Artist: BTS
Album: Love Yourself 承 'Her'
Genre: k-pop
Popularity: 75
----------------------------------------
Track: Born Singer
Artist: BTS
Album: Proof
Genre: k-pop
Popularity: 75
----------------------------------------
Track: Yet To Come

## 13. Artist-Specific Exploration

Look more closely at The Weeknd tracks and filter them by popularity.


In [30]:
df[df["artists"].str.contains("The Weeknd", case=False, na=False)]

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,track_genre
3553,4hrfgsdCAOW9GYDcpNfCWv,FKA twigs;The Weeknd,Music for Rainy Days,tears in the club,0,196760,True,0.616,0.717,8,-6.178,1,0.1100,0.194,0.000098,0.0731,0.720,167.990,alternative
3556,1djJRLSBe7UKJ7F1p3tidC,FKA twigs;The Weeknd,New Grooves,tears in the club,0,196760,True,0.616,0.717,8,-6.178,1,0.1100,0.194,0.000098,0.0731,0.720,167.990,alternative
3557,2M5t4tdhZ9W3Eys2HDCjbY,FKA twigs;The Weeknd,On Chill - Rap & RnB,tears in the club,0,196760,True,0.616,0.717,8,-6.178,1,0.1100,0.194,0.000098,0.0731,0.720,167.990,alternative
3567,6BrLfEagscVPCOxQa0uNxG,FKA twigs;The Weeknd,20s Love Songs,tears in the club,1,196760,True,0.616,0.717,8,-6.178,1,0.1100,0.194,0.000098,0.0731,0.720,167.990,alternative
3568,4HU70kqhA6tRYl55kUkg0a,FKA twigs;The Weeknd,Good Vibes,tears in the club,0,196760,True,0.616,0.717,8,-6.178,1,0.1100,0.194,0.000098,0.0731,0.720,167.990,alternative
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111019,1djJRLSBe7UKJ7F1p3tidC,FKA twigs;The Weeknd,New Grooves,tears in the club,0,196760,True,0.616,0.717,8,-6.178,1,0.1100,0.194,0.000098,0.0731,0.720,167.990,trip-hop
111020,2M5t4tdhZ9W3Eys2HDCjbY,FKA twigs;The Weeknd,On Chill - Rap & RnB,tears in the club,0,196760,True,0.616,0.717,8,-6.178,1,0.1100,0.194,0.000098,0.0731,0.720,167.990,trip-hop
111027,4HU70kqhA6tRYl55kUkg0a,FKA twigs;The Weeknd,Good Vibes,tears in the club,0,196760,True,0.616,0.717,8,-6.178,1,0.1100,0.194,0.000098,0.0731,0.720,167.990,trip-hop
111031,3WB7O03A1THXLWrtAERC3h,FKA twigs;The Weeknd,Hit After Hit,tears in the club,0,196760,True,0.616,0.717,8,-6.178,1,0.1100,0.194,0.000098,0.0731,0.720,167.990,trip-hop


In [31]:
query = {
    "bool": {
        "must": [
            {"match": {"artists": "The Weeknd"}}
        ],
        "filter": [
            {"range": {"popularity": {"gte": 60}}}
        ]
    }
}

results = client.search(index=INDEX_NAME, query=query)
count = client.count(index=INDEX_NAME)["count"]
print(f"Total songs found: {count}")
print("-" * 60)
filter_count = client.count(index=INDEX_NAME, query=query)["count"]
print(f"Songs matching filter: {filter_count}")
print("-" * 60)

for hit in results["hits"]["hits"]:
    song = hit["_source"]

    print(f"Track: {song['track_name']}")
    print(f"Artist: {song['artists']}")
    print(f"Album: {song['album_name']}")
    print(f"Genre: {song['track_genre']}")
    print(f"Popularity: {song['popularity']}")
    print("-" * 40)



Total songs found: 89740
------------------------------------------------------------
Songs matching filter: 738
------------------------------------------------------------
Track: Blinding Lights
Artist: The Weeknd
Album: After Hours
Genre: pop
Popularity: 91
----------------------------------------
Track: Call Out My Name
Artist: The Weeknd
Album: My Dear Melancholy,
Genre: pop
Popularity: 89
----------------------------------------
Track: Save Your Tears
Artist: The Weeknd
Album: After Hours
Genre: pop
Popularity: 89
----------------------------------------
Track: The Hills
Artist: The Weeknd
Album: Beauty Behind The Madness
Genre: pop
Popularity: 88
----------------------------------------
Track: Die For You
Artist: The Weeknd
Album: Starboy
Genre: pop
Popularity: 88
----------------------------------------
Track: Make It (feat. The Weeknd) [DJAmg Remixer]
Artist: AtariJones;The Weeknd
Album: Early Catch
Genre: chill
Popularity: 62
----------------------------------------
Track: I 

## 14. Album Aggregations

Use Elasticsearch aggregations to group an artist's tracks by album and calculate average popularity per album.


In [32]:
query ={
    "bool": {
        "must": [
            {"match": {"artists": "The Weeknd"}}
        ]
    }
}

aggs = {
    "albums": {
        "terms": {
            "field": "album_name.keyword",
            "order": {"avg_popularity": "desc"},
        },
        "aggs": {
            "avg_popularity": {
                "avg": {
                    "field": "popularity"
                }
            }
        }
    }
}

results = client.search(
    index=INDEX_NAME, 
    query=query, 
    aggs=aggs, 
    size=0)

count = client.count(index=INDEX_NAME)["count"]
print(f"Total songs found: {count}")
print("-" * 60)
filter_count = client.count(index=INDEX_NAME, query=query)["count"]
print(f"Songs matching filter: {filter_count}")
print("-" * 60)

for bucket in results["aggregations"]["albums"]["buckets"]:
    album_name = bucket["key"]
    doc_count = bucket["doc_count"]
    avg_popularity = bucket["avg_popularity"]["value"]

    print(f"Album: {album_name}")
    print(f"Number of Songs: {doc_count}")
    print(f"Average Popularity: {avg_popularity:.2f}")
    print("-" * 40)



Total songs found: 89740
------------------------------------------------------------
Songs matching filter: 5345
------------------------------------------------------------
Album: After Hours
Number of Songs: 2
Average Popularity: 90.00
----------------------------------------
Album: My Dear Melancholy,
Number of Songs: 2
Average Popularity: 89.50
----------------------------------------
Album: I Love You So
Number of Songs: 1
Average Popularity: 89.00
----------------------------------------
Album: STAY (with Justin Bieber)
Number of Songs: 1
Average Popularity: 89.00
----------------------------------------
Album: Starboy
Number of Songs: 2
Average Popularity: 89.00
----------------------------------------
Album: Beauty Behind The Madness
Number of Songs: 1
Average Popularity: 88.00
----------------------------------------
Album: F*CK LOVE 3: OVER YOU
Number of Songs: 1
Average Popularity: 86.00
----------------------------------------
Album: Synchronicity (Remastered 2003)
Number 

## 15. Inspect Aggregation Output

Print the raw hit total and aggregation object for a final low-level check.


In [33]:
print(results["hits"]["total"]["value"])
print(results["aggregations"])

5345
{'albums': {'doc_count_error_upper_bound': -1, 'sum_other_doc_count': 5332, 'buckets': [{'key': 'After Hours', 'doc_count': 2, 'avg_popularity': {'value': 90.0}}, {'key': 'My Dear Melancholy,', 'doc_count': 2, 'avg_popularity': {'value': 89.5}}, {'key': 'I Love You So', 'doc_count': 1, 'avg_popularity': {'value': 89.0}}, {'key': 'STAY (with Justin Bieber)', 'doc_count': 1, 'avg_popularity': {'value': 89.0}}, {'key': 'Starboy', 'doc_count': 2, 'avg_popularity': {'value': 89.0}}, {'key': 'Beauty Behind The Madness', 'doc_count': 1, 'avg_popularity': {'value': 88.0}}, {'key': 'F*CK LOVE 3: OVER YOU', 'doc_count': 1, 'avg_popularity': {'value': 86.0}}, {'key': 'Synchronicity (Remastered 2003)', 'doc_count': 1, 'avg_popularity': {'value': 86.0}}, {'key': 'Death of a Bachelor', 'doc_count': 1, 'avg_popularity': {'value': 85.0}}, {'key': 'Un Verano Sin Ti', 'doc_count': 1, 'avg_popularity': {'value': 85.0}}]}}
